In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CARGA DESDE SILVER - ÚLTIMO data_ingestion_ts
# ============================================================

print("=" * 60)
print("CARGA DESDE SILVER")
print("=" * 60)

table_silver = "adbsmartdatamanuelestrada.silver.quality_layer_slv"

# 1. Obtener el último timestamp de ingestión
ultimo_ts = spark.table(table_silver) \
    .select(F.max("data_ingestion_ts").alias("max_ts")) \
    .collect()[0]["max_ts"]

print(f"\nÚltimo data_ingestion_ts en Silver: {ultimo_ts}")

# 2. Cargar solo los datos del último timestamp
df_silver = spark.table(table_silver) \
    .filter(F.col("data_ingestion_ts") == ultimo_ts)

registros_cargados = df_silver.count()

print(f"\nRegistros cargados: {registros_cargados:,}")
print(f"Columnas: {len(df_silver.columns)}")
print("\nEsquema:")
df_silver.printSchema()
print("=" * 60)

In [0]:
# ============================================================
# ENRIQUECIMIENTO - MÉTRICAS CALCULADAS
# ============================================================

print("=" * 60)
print("ENRIQUECIMIENTO - MÉTRICAS CALCULADAS")
print("=" * 60)

print("\nCalculando métricas de negocio...")

# Crear todas las métricas calculadas en secuencia (redondeadas a 2 decimales)
# data_ingestion_ts se hereda automáticamente como timestamp desde Silver
df_gold = df_silver \
    .withColumn("venta", 
                F.round(F.col("cantidad") * F.col("precio_unit"), 2)) \
    .withColumn("coste_venta", 
                F.round(F.col("cantidad") * F.col("coste_unit"), 2)) \
    .withColumn("iva", 
                F.round(F.col("venta") * (F.col("porc_iva") / 100), 2)) \
    .withColumn("ventasiniva", 
                F.round(F.col("venta") - F.col("iva"), 2)) \
    .withColumn("margen", 
                F.round(F.col("ventasiniva") - F.col("coste_venta"), 2)) \
    .withColumn("porc_margen", 
                F.round(F.col("margen") / F.col("ventasiniva") * 100, 2)) \
    .select(
        "fecha_orden",
        "cod_venta",
        "cod_tienda",
        "cod_producto",
        "cod_cliente",
        "cantidad",
        "precio_unit",
        "coste_unit",
        "porc_iva",
        "venta",
        "coste_venta",
        "iva",
        "ventasiniva",
        "margen",
        "porc_margen",
        "data_ingestion_ts"  # ← AL FINAL
    )

print("\n✅ Métricas calculadas:")
print("  1. venta = cantidad * precio_unit")
print("  2. coste_venta = cantidad * coste_unit")
print("  3. iva = venta * (porc_iva / 100)")
print("  4. ventasiniva = venta - iva")
print("  5. margen = ventasiniva - coste_venta")
print("  6. porc_margen = margen / ventasiniva * 100")

print(f"\nRegistros: {df_gold.count():,}")
print(f"Columnas totales: {len(df_gold.columns)} (original: {len(df_silver.columns)}, añadidas: 6)")

# Vista previa de las métricas calculadas
print("\nVista previa de métricas calculadas (primeras 5 filas):")
display(df_gold.select(
    "fecha_orden",
    "cod_venta",
    "cantidad",
    "precio_unit",
    "coste_unit",
    "porc_iva",
    "venta",
    "coste_venta",
    "iva",
    "ventasiniva",
    "margen",
    "porc_margen"
).limit(5))

# Resumen de métricas totales
print("\nResumen de métricas totales:")
df_gold.select(
    F.sum("venta").alias("venta_total"),
    F.sum("coste_venta").alias("coste_total"),
    F.sum("iva").alias("iva_total"),
    F.sum("ventasiniva").alias("ventasiniva_total"),
    F.sum("margen").alias("margen_total"),
    F.avg("porc_margen").alias("porc_margen_promedio")
).show()

print("=" * 60)

In [0]:
# ============================================================
# CONTROL DE DUPLICADOS POR MES - GOLD LAYER
# ============================================================

print("=" * 60)
print("CONTROL DE DUPLICADOS POR MES")
print("=" * 60)

table_gold = "adbsmartdatamanuelestrada.gold.semantic_layer_gld"

# 1. Extraer el mes de los datos actuales en df_gold
df_con_mes = df_gold.withColumn(
    "mes_registro",
    F.concat(
        F.year("fecha_orden"),
        F.lpad(F.month("fecha_orden"), 2, "0")
    )
)

# Obtener el mes del DataFrame actual
meses_en_datos = df_con_mes.select("mes_registro").distinct().collect()
meses_carga = [row['mes_registro'] for row in meses_en_datos]
print(f"\nMes(es) en los datos a cargar: {', '.join(meses_carga)}")

if len(meses_carga) > 1:
    raise ValueError(
        f"❌ ERROR: Los datos contienen múltiples meses: {', '.join(meses_carga)}\n"
        f"   La carga incremental debe ser de un solo mes a la vez."
    )

mes_actual = meses_carga[0]
print(f"Mes de la carga: {mes_actual}")

# 2. Verificar si el mes ya existe en la tabla Gold
try:
    # Intentar leer la tabla existente
    tabla_existente = spark.table(table_gold)
    
    # Extraer meses existentes en la tabla
    meses_existentes = tabla_existente.select(
        F.concat(
            F.year("fecha_orden"),
            F.lpad(F.month("fecha_orden"), 2, "0")
        ).alias("mes_tabla")
    ).distinct().collect()
    
    meses_en_tabla = [row['mes_tabla'] for row in meses_existentes]
    print(f"\nMeses ya existentes en tabla Gold: {', '.join(meses_en_tabla)}")
    
    if mes_actual in meses_en_tabla:
        print(f"\n⚠️  ADVERTENCIA: El mes {mes_actual} ya existe en la tabla Gold.")
        print(f"   No se realizará la inserción para evitar duplicados.")
        puede_appendear = False
    else:
        print(f"\n✅ El mes {mes_actual} NO existe en la tabla. Se procederá a appendear.")
        puede_appendear = True
        
except Exception as e:
    # La tabla no existe, es la primera carga
    print(f"\nℹ️  La tabla Gold no existe aún. Se creará con los datos del mes {mes_actual}.")
    puede_appendear = True

print("\n" + "=" * 60)

In [0]:
# ============================================================
# ESCRITURA A TABLA GOLD (INCREMENTAL)
# ============================================================

print("=" * 60)
print("ESCRITURA A TABLA GOLD")
print("=" * 60)

# Escribir a la tabla Gold (modo incremental)
if puede_appendear:
    # Remover columna temporal antes de escribir
    df_final = df_con_mes.drop("mes_registro")
    
    registros_a_insertar = df_final.count()
    print(f"\nRegistros a insertar: {registros_a_insertar:,}")
    print(f"Columnas: {len(df_final.columns)}")
    
    df_final.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(table_gold)
    
    print("\n" + "=" * 60)
    print("✅ INSERCIÓN EXITOSA")
    print("=" * 60)
    print(f"Tabla: {table_gold}")
    print(f"Mes insertado: {mes_actual}")
    print(f"Registros insertados: {registros_a_insertar:,}")
    print(f"Total registros en tabla: {spark.table(table_gold).count():,}")
    print(f"Modo: APPEND (incremental)")
    print("=" * 60)
else:
    print("\n" + "=" * 60)
    print("⛔ OPERACIÓN CANCELADA")
    print("=" * 60)
    print(f"No se insertaron registros para evitar duplicados del mes {mes_actual}")
    print(f"Total registros en tabla (sin cambios): {spark.table(table_gold).count():,}")
    print("=" * 60)